# Sales Transactions Cleaning — El-Sharnouby Group


In [2]:
# ============================================================
# 0.1 Imports, display settings, helper functions
# ============================================================
import pandas as pd
import numpy as np
import re
import time
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print("pandas version:", pd.__version__)


def get_text_cols(df):
    """Text columns in pandas 2 ("object") and pandas 3 ("str")."""
    return [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]


def read_csv_safe(path):
    """Read a CSV trying UTF-8 (with/without BOM) first, then Windows Arabic."""
    for enc in ["utf-8-sig", "cp1256"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()
            return df, enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not read {path} with any known encoding")


DATE_FORMATS = ["%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"]

def parse_dates(series, name=""):
    """Try each date format; keep the one that converts the most values."""
    best_result, best_fmt, best_ok = None, None, -1
    for fmt in DATE_FORMATS:
        converted = pd.to_datetime(series, format=fmt, errors="coerce")
        ok = converted.notna().sum()
        if ok > best_ok:
            best_result, best_fmt, best_ok = converted, fmt, ok
    filled = series.notna().sum()
    failed = filled - best_ok
    print(f"[{name}] format used: {best_fmt}  ->  converted {best_ok:,} of {filled:,}  (failed: {failed:,})")
    if failed > 0:
        print("   examples that failed:", series[series.notna() & best_result.isna()].head().tolist())
    return best_result

pandas version: 3.0.6


In [3]:
# ============================================================
# 0.2 Paths + load CLEAN reference files (from notebook 01)
# ============================================================
RAW_DIR   = Path("../data/raw")
CLEAN_DIR = Path("../data/clean")

# We read the CLEANED versions (parquet), not the raw CSVs:
# the sales file will be checked against them.
prod = pd.read_parquet(CLEAN_DIR / "products.parquet")
loc  = pd.read_parquet(CLEAN_DIR / "locations.parquet")
cust = pd.read_parquet(CLEAN_DIR / "customers.parquet")

print(f"products:  {prod.shape}")
print(f"locations: {loc.shape}")
print(f"customers: {cust.shape}")
print("\nJoin_Date type:", cust["Join_Date"].dtype, " <- parquet kept the date type")

products:  (59, 10)
locations: (5, 9)
customers: (30000, 8)

Join_Date type: datetime64[us]  <- parquet kept the date type


In [4]:
# ============================================================
# 1.1 Load Sales (large file — this takes a few seconds)
# ============================================================
t0 = time.time()
sales_raw, enc = read_csv_safe(RAW_DIR / "Sales_Transactions.csv")
print(f"Loaded in {time.time() - t0:.1f} s  (encoding: {enc})")

sales = sales_raw.copy()   # sales_raw stays untouched as the original

print(f"\nShape: {sales.shape[0]:,} rows x {sales.shape[1]} cols")
print(f"Memory: {sales.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("\nAll columns:", list(sales.columns))

print("\nRaw date examples:")
for col in ["Order_Date", "Return_Date", "Month_Key"]:
    print(f"  {col}: {sales[col].dropna().head(3).tolist()}")

sales.head(5)

Loaded in 2.5 s  (encoding: utf-8-sig)

Shape: 180,000 rows x 30 cols
Memory: 80.8 MB

All columns: ['Row_ID', 'Invoice_ID', 'Order_Date', 'Month_Key', 'Location_ID', 'Location_Name', 'Channel', 'Customer_ID', 'Customer_Type', 'Customer_Governorate', 'Product_ID', 'SKU', 'Product_Name', 'Product_Family', 'Size', 'Quantity', 'List_Price_EGP', 'Discount_Pct', 'Gross_Sales_EGP', 'Net_Sales_EGP', 'Estimated_COGS_EGP', 'Gross_Profit_EGP', 'Gross_Margin_Pct', 'Payment_Method', 'Returned', 'Return_Date', 'Return_Reason', 'Refund_Value_EGP', 'Quality_Related_Return', 'Data_Class']

Raw date examples:
  Order_Date: ['2024-05-11', '2025-07-31', '2026-05-23']
  Return_Date: ['2024-10-09', '2023-11-23', '2024-09-25']
  Month_Key: ['2024-05', '2025-07', '2026-05']


,Row_ID,Invoice_ID,Order_Date,Month_Key,Location_ID,Location_Name,Channel,Customer_ID,Customer_Type,Customer_Governorate,Product_ID,SKU,Product_Name,Product_Family,Size,Quantity,List_Price_EGP,Discount_Pct,Gross_Sales_EGP,Net_Sales_EGP,Estimated_COGS_EGP,Gross_Profit_EGP,Gross_Margin_Pct,Payment_Method,Returned,Return_Date,Return_Reason,Refund_Value_EGP,Quality_Related_Return,Data_Class
0,0,INV0000001,2024-05-11,2024-05,LOC002,LEO MODA Online Store,Online,C18976,Retail,Cairo,P059,CAT-059,Positive vibes - 2024بيجامة حريمي صيفي,Pajamas/Homewear,M,1,388,0.15,388,329.80,247.57,82.23,0.2493,Wallet,No,NaN,No return,0.0,No,Modeled training transaction anchored to public products...
1,1,INV0000002,2025-07-31,2025-07,LOC001,LEO MODA - Sidi Bishr Store,Retail,C22543,Retail,Alexandria,P038,CAT-038,Boys' Autumn Pajamas-Mickey Mouse,Pajamas/Homewear,8,1,289,0.20,289,231.20,163.31,67.89,0.2936,Card,No,NaN,No return,0.0,No,Modeled training transaction anchored to public products...
2,2,INV0000003,2026-05-23,2026-05,LOC003,El-Sharnouby Wholesale Desk,Wholesale,C13351,Wholesale,Alexandria,P028,CAT-028,Girls' summer pajamas,Pajamas/Homewear,6,24,295,0.32,7080,4814.40,4301.71,512.69,0.1065,Cash,No,NaN,No return,0.0,No,Modeled training transaction anchored to public products...
3,3,INV0000004,2024-06-05,2024-06,LOC003,El-Sharnouby Wholesale Desk,Wholesale,C21377,Wholesale,Dakahlia,P004,26115-7,Summer children's girls-Hello kitty black&white,Pajamas/Homewear,6,48,348,0.32,16704,11358.72,10451.13,907.59,0.0799,Bank Transfer,No,NaN,No return,0.0,No,Modeled training transaction anchored to public products...
4,4,INV0000005,2025-12-28,2025-12,LOC001,LEO MODA - Sidi Bishr Store,Retail,C21499,Retail,Alexandria,P024,CAT-024,Boys' summer pajamas-Mickey,Pajamas/Homewear,12,1,299,0.20,299,239.20,186.16,53.04,0.2217,Cash,No,NaN,No return,0.0,No,Modeled training transaction anchored to public products...


In [5]:
# ============================================================
# 1.2 Types, missing values, duplicates
# ============================================================
sales.info()

print("\nMissing values per column (only columns with missing):")
nulls = sales.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "None")

# Row_ID is a unique counter -> a "full duplicate" check must IGNORE it,
# otherwise two identical sales lines would never look duplicated
print("\nDuplicated Row_ID:", sales["Row_ID"].duplicated().sum())
dup_rows = sales.drop(columns=["Row_ID"]).duplicated()
print("Fully duplicated rows (ignoring Row_ID):", dup_rows.sum())

# Same product + same size twice in the SAME invoice -> unusual, report only
dup_line = sales.duplicated(subset=["Invoice_ID", "Product_ID", "Size"], keep=False)
print("Lines repeating (Invoice, Product, Size):", dup_line.sum())

# Invoice structure
lines_per_invoice = sales.groupby("Invoice_ID").size()
print(f"\nInvoices: {sales['Invoice_ID'].nunique():,}")
print("Lines per invoice:")
print(lines_per_invoice.describe().round(2))

<class 'pandas.DataFrame'>
RangeIndex: 180000 entries, 0 to 179999
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Row_ID                  180000 non-null  int64  
 1   Invoice_ID              180000 non-null  str    
 2   Order_Date              180000 non-null  str    
 3   Month_Key               180000 non-null  str    
 4   Location_ID             180000 non-null  str    
 5   Location_Name           180000 non-null  str    
 6   Channel                 180000 non-null  str    
 7   Customer_ID             180000 non-null  str    
 8   Customer_Type           180000 non-null  str    
 9   Customer_Governorate    180000 non-null  str    
 10  Product_ID              180000 non-null  str    
 11  SKU                     180000 non-null  str    
 12  Product_Name            180000 non-null  str    
 13  Product_Family          180000 non-null  str    
 14  Size                    180000 

In [6]:
# ============================================================
# 1.3 Distinct values in categorical columns
# ============================================================
category_cols = ["Location_ID", "Location_Name", "Channel", "Customer_Type",
                 "Customer_Governorate", "Product_Family", "Size", "Payment_Method",
                 "Returned", "Return_Reason", "Quality_Related_Return", "Data_Class"]

for col in category_cols:
    print(f"\n========== {col} ({sales[col].nunique()} distinct) ==========")
    print(sales[col].value_counts(dropna=False).head(25))


========== Location_ID (3 distinct) ==========
Location_ID
LOC002    71559
LOC001    61769
LOC003    46672
Name: count, dtype: int64

========== Location_Name (3 distinct) ==========
Location_Name
LEO MODA Online Store          71559
LEO MODA - Sidi Bishr Store    61769
El-Sharnouby Wholesale Desk    46672
Name: count, dtype: int64

========== Channel (3 distinct) ==========
Channel
Online       71559
Retail       61769
Wholesale    46672
Name: count, dtype: int64

========== Customer_Type (2 distinct) ==========
Customer_Type
Retail       135053
Wholesale     44947
Name: count, dtype: int64

========== Customer_Governorate (13 distinct) ==========
Customer_Governorate
Alexandria        51597
Cairo             31864
Giza              20000
Beheira           14628
Dakahlia          11410
Gharbia           10445
Other             10240
Sharqia            8932
Kafr El Sheikh     7240
Monufia            4988
Port Said          3440
Damietta           3440
Suez               1776
Name: cou

In [7]:
# ============================================================
# 1.4 Strip spaces + drop Row_ID and Data_Class
# ============================================================
t0 = time.time()
for col in get_text_cols(sales):
    sales[col] = sales[col].str.strip()
print(f"Stripped {len(get_text_cols(sales))} text columns in {time.time() - t0:.1f} s")

# Row_ID   -> just a counter (0, 1, 2 ...), no business meaning
# Data_Class -> constant sentence
print("Data_Class distinct values:", sales["Data_Class"].nunique())
sales = sales.drop(columns=["Row_ID", "Data_Class"], errors="ignore")

print("Remaining columns:", len(sales.columns))

Stripped 20 text columns in 0.2 s
Data_Class distinct values: 1
Remaining columns: 28


In [8]:
# ============================================================
# 1.5 Dates: Order_Date, Return_Date, and Month_Key consistency
# ============================================================
sales["Order_Date"]  = parse_dates(sales_raw["Order_Date"],  name="Order_Date")
sales["Return_Date"] = parse_dates(sales_raw["Return_Date"], name="Return_Date")

first, last = sales["Order_Date"].min(), sales["Order_Date"].max()
print(f"\nOrder_Date range: {first.date()} -> {last.date()}")
print("Orders dated in the future:", (sales["Order_Date"] > pd.Timestamp.today().normalize()).sum())

# 1) Month_Key must match the month of Order_Date (e.g. 2024-05 for 2024-05-11)
month_from_date = sales["Order_Date"].dt.strftime("%Y-%m")
month_mismatch = month_from_date != sales["Month_Key"]
print(f"\nMonth_Key not matching Order_Date: {month_mismatch.sum():,}")

# 2) A return can't happen before the order
returned_early = sales["Return_Date"] < sales["Order_Date"]
print(f"Return_Date before Order_Date:     {returned_early.sum():,}")

# 3) Return_Date should exist ONLY for returned lines
print("\nReturned flag vs. Return_Date filled:")
print(pd.crosstab(sales["Returned"], sales["Return_Date"].notna(), margins=True))

# 4) How long after the order do returns happen?
days_to_return = (sales["Return_Date"] - sales["Order_Date"]).dt.days
print("\nDays from order to return:")
print(days_to_return.describe().round(1))

print("\nLines per year:")
print(sales["Order_Date"].dt.year.value_counts().sort_index())

[Order_Date] format used: %Y-%m-%d  ->  converted 180,000 of 180,000  (failed: 0)
[Return_Date] format used: %Y-%m-%d  ->  converted 8,543 of 8,543  (failed: 0)

Order_Date range: 2023-10-01 -> 2026-09-30
Orders dated in the future: 0

Month_Key not matching Order_Date: 0
Return_Date before Order_Date:     0

Returned flag vs. Return_Date filled:
Return_Date   False  True     All
Returned                         
No           171457     0  171457
Yes               0  8543    8543
All          171457  8543  180000

Days from order to return:
count    8543.0
mean       10.8
std         6.1
min         0.0
25%         6.0
50%        11.0
75%        16.0
max        21.0
dtype: float64

Lines per year:
Order_Date
2023    12810
2024    55260
2025    60735
2026    51195
Name: count, dtype: int64


In [9]:
# ============================================================
# 1.6 Referential integrity: every ID must exist in its reference file
# ============================================================
# If an ID is missing from its reference file, those sales lines will
# "disappear" (or get empty attributes) when we join tables later.
checks = {
    "Product_ID":  (sales["Product_ID"],  prod["Product_ID"]),
    "Location_ID": (sales["Location_ID"], loc["Location_ID"]),
    "Customer_ID": (sales["Customer_ID"], cust["Customer_ID"]),
}
for name, (in_sales, in_ref) in checks.items():
    missing = set(in_sales) - set(in_ref)
    lines   = in_sales.isin(missing).sum()
    print(f"{name:<12} missing from reference: {len(missing):>5} IDs  ({lines:,} sales lines)")
    if missing:
        print("   examples:", sorted(missing)[:10])

# Which locations actually sell?
print("\nSales lines per location:")
loc_names = loc.set_index("Location_ID")["Location_Short"]
print(sales["Location_ID"].map(loc_names).value_counts())

# Customers who never bought anything
buyers = set(sales["Customer_ID"])
never_bought = cust[~cust["Customer_ID"].isin(buyers)]
print(f"\nCustomers with NO purchases: {len(never_bought):,}")
print(never_bought["Customer_Type"].value_counts())

# Sales BEFORE the customer joined (impossible)
join_date = sales["Customer_ID"].map(cust.set_index("Customer_ID")["Join_Date"])
before_join = sales["Order_Date"] < join_date
print(f"\nSales dated before the customer's Join_Date: {before_join.sum():,}")
if before_join.any():
    gap_days = (join_date - sales["Order_Date"]).dt.days[before_join]
    print("   days before join:", gap_days.describe().round(1).to_dict())

Product_ID   missing from reference:     0 IDs  (0 sales lines)
Location_ID  missing from reference:     0 IDs  (0 sales lines)
Customer_ID  missing from reference:     0 IDs  (0 sales lines)

Sales lines per location:
Location_ID
Online Store        71559
Sidi Bishr Store    61769
Wholesale Desk      46672
Name: count, dtype: int64

Customers with NO purchases: 4,366
Customer_Type
Retail       4314
Wholesale      52
Name: count, dtype: int64

Sales dated before the customer's Join_Date: 0


In [10]:
# ============================================================
# 1.7 Do duplicated descriptive columns agree with the reference files?
# ============================================================
# Sales repeats some attributes that already live in reference files.
# Before dropping them (normalization), check they agree.
# .astype(str) on both sides -> compare plain text (category vs text can't be compared directly)
def compare(sales_col, ref_df, ref_key, ref_col, sales_key):
    ref_map  = ref_df.set_index(ref_key)[ref_col].astype(str)
    expected = sales[sales_key].map(ref_map)
    actual   = sales[sales_col].astype(str)
    mismatch = expected.notna() & (expected != actual)
    print(f"{sales_col:<22} vs {ref_col:<18}: {mismatch.sum():>8,} mismatches")
    return mismatch

m_type = compare("Customer_Type",        cust, "Customer_ID", "Customer_Type", "Customer_ID")
m_gov  = compare("Customer_Governorate", cust, "Customer_ID", "Governorate",   "Customer_ID")
m_loc  = compare("Location_Name",        loc,  "Location_ID", "Location_Name", "Location_ID")
m_fam  = compare("Product_Family",       prod, "Product_ID",  "Product_Family","Product_ID")
m_sku  = compare("SKU",                  prod, "Product_ID",  "SKU",           "Product_ID")
m_name = compare("Product_Name",         prod, "Product_ID",  "Product_Name",  "Product_ID")

# Product_Name: we tidied spaces in the catalog ("صيفي2024" -> "صيفي 2024"),
# so some differences are EXPECTED. Show a few to confirm that's the only reason.
if m_name.any():
    examples = sales.loc[m_name, ["Product_ID", "Product_Name"]].drop_duplicates("Product_ID").head(5)
    examples["Catalog_Name"] = examples["Product_ID"].map(prod.set_index("Product_ID")["Product_Name"])
    display(examples)

Customer_Type          vs Customer_Type     :        0 mismatches
Customer_Governorate   vs Governorate       :        0 mismatches
Location_Name          vs Location_Name     :        0 mismatches
Product_Family         vs Product_Family    :        0 mismatches
SKU                    vs SKU               :        0 mismatches
Product_Name           vs Product_Name      :   25,570 mismatches


,Product_ID,Product_Name,Catalog_Name
0,P059,Positive vibes - 2024بيجامة حريمي صيفي,Positive vibes - 2024 بيجامة حريمي صيفي
13,P054,Gazelle - بيجامة حريمي صيفي2024,Gazelle - بيجامة حريمي صيفي 2024
53,P055,Scooby-Doo - بيجامة حريمي صيفي2024,Scooby-Doo - بيجامة حريمي صيفي 2024
67,P051,Ice cream - بيجامة حريمي صيفي2024,Ice cream - بيجامة حريمي صيفي 2024
73,P056,BATGIRL - بيجامة حريمي صيفي2024,BATGIRL - بيجامة حريمي صيفي 2024


In [11]:
# ============================================================
# 2.1 Consistency: Channel vs Location vs Customer_Type
# ============================================================
# Expected: Wholesale channel <-> Wholesale Desk <-> Wholesale customers
#           Retail channel    <-> Sidi Bishr Store
#           Online channel    <-> Online Store
loc_short = sales["Location_ID"].map(loc.set_index("Location_ID")["Location_Short"])

print("========== Channel x Location ==========")
print(pd.crosstab(sales["Channel"], loc_short))

print("\n========== Channel x Customer_Type ==========")
print(pd.crosstab(sales["Channel"], sales["Customer_Type"]))

print("\n========== Payment_Method x Channel ==========")
print(pd.crosstab(sales["Payment_Method"], sales["Channel"]))

========== Channel x Location ==========
Location_ID  Online Store  Sidi Bishr Store  Wholesale Desk
Channel                                                    
Online              71559                 0               0
Retail                  0             61769               0
Wholesale               0                 0           46672

========== Channel x Customer_Type ==========
Customer_Type  Retail  Wholesale
Channel                         
Online          71432        127
Retail          61660        109
Wholesale        1961      44711

========== Payment_Method x Channel ==========
Channel         Online  Retail  Wholesale
Payment_Method                           
Bank Transfer     5577    4958      34940
Card             23688   20467       2323
Cash             30306   25819       8497
Wallet           11988   10525        912


In [12]:
# ============================================================
# 2.2 Confirm Product_Name differences are ONLY spacing, then drop duplicated columns
# ============================================================
# Same tidy function used on the catalog in notebook 01
ARABIC = "\u0600-\u06FF"
def tidy_name(text):
    if pd.isna(text):
        return text
    s = str(text)
    s = re.sub(f"([{ARABIC}])(\\d)", r"\1 \2", s)   # Arabic letter then digit -> add space
    s = re.sub(f"(\\d)([{ARABIC}])", r"\1 \2", s)   # digit then Arabic letter -> add space
    s = re.sub(r"\s+", " ", s).strip()             # collapse multiple spaces
    return s

# Tidy the distinct names only (59 products) — much faster than 180,000 rows
name_map = {n: tidy_name(n) for n in sales["Product_Name"].unique()}
tidied = sales["Product_Name"].map(name_map)
catalog_name = sales["Product_ID"].map(prod.set_index("Product_ID")["Product_Name"])
print("Product_Name mismatches AFTER tidying:", (tidied != catalog_name).sum(), "(must be 0)")

# All verified as identical to the reference files -> drop (normalization)
# Month_Key           -> = Order_Date month (verified in 1.5)
# Location_Name       -> locations   (verified in 1.7)
# Customer_Type       -> customers   (verified in 1.7)
# Customer_Governorate-> customers   (verified in 1.7)
# Product_Family, SKU, Product_Name -> products (verified in 1.7 / above)
redundant = ["Month_Key", "Location_Name", "Customer_Type", "Customer_Governorate",
             "Product_Family", "SKU", "Product_Name"]
sales = sales.drop(columns=redundant, errors="ignore")
print("\nRemaining columns:", len(sales.columns))
print(list(sales.columns))

Product_Name mismatches AFTER tidying: 0 (must be 0)

Remaining columns: 21
['Invoice_ID', 'Order_Date', 'Location_ID', 'Channel', 'Customer_ID', 'Product_ID', 'Size', 'Quantity', 'List_Price_EGP', 'Discount_Pct', 'Gross_Sales_EGP', 'Net_Sales_EGP', 'Estimated_COGS_EGP', 'Gross_Profit_EGP', 'Gross_Margin_Pct', 'Payment_Method', 'Returned', 'Return_Date', 'Return_Reason', 'Refund_Value_EGP', 'Quality_Related_Return']


In [13]:
# ============================================================
# 2.3 Sizes: valid for the product? consistent with the audience?
# ============================================================
sales["Size"] = sales["Size"].astype(str).str.strip().str.upper()

print("Size values:")
print(sales["Size"].value_counts().sort_index())

# Build the list of VALID (product, size) pairs from the catalog
# explode -> one row per size: P001 [4,6,8] becomes P001-4, P001-6, P001-8
valid_pairs = (prod[["Product_ID", "Sizes_List"]]
               .explode("Sizes_List")
               .rename(columns={"Sizes_List": "Size"}))
valid_pairs["Size"] = valid_pairs["Size"].astype(str)
valid_pairs["Is_Valid_Size"] = True

# Left merge keeps every sales row; rows with no match get NaN in Is_Valid_Size
check = sales[["Product_ID", "Size"]].merge(valid_pairs, on=["Product_ID", "Size"], how="left")
invalid = check["Is_Valid_Size"].isna()
print(f"\nSales lines with a size NOT offered for that product: {invalid.sum():,}")
if invalid.any():
    print(check.loc[invalid, ["Product_ID", "Size"]].value_counts().head(10))

# Size system vs audience: Kids = numbers, Women = letters
audience = sales["Product_ID"].map(prod.set_index("Product_ID")["Audience"])
is_numeric_size = sales["Size"].str.isdigit()
print("\n========== Audience x numeric size ==========")
print(pd.crosstab(audience, is_numeric_size.map({True: "Numeric (Kids)", False: "Letter (Adult)"})))

Size values:
Size
10     30355
12     29934
2XL     7348
4      30144
6      29918
8      30304
L       7316
M       7283
XL      7398
Name: count, dtype: int64

Sales lines with a size NOT offered for that product: 0

========== Audience x numeric size ==========
Size        Letter (Adult)  Numeric (Kids)
Product_ID                                
Kids                     0          150655
Women                29345               0


In [14]:
# ============================================================
# 2.4 Numeric sanity
# ============================================================
money_cols = ["List_Price_EGP", "Gross_Sales_EGP", "Net_Sales_EGP",
              "Estimated_COGS_EGP", "Gross_Profit_EGP", "Refund_Value_EGP"]
for col in ["Quantity", "Discount_Pct", "Gross_Margin_Pct"] + money_cols:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

print("Quantity <= 0:            ", (sales["Quantity"] <= 0).sum())
print("Quantity not whole number:", (sales["Quantity"] % 1 != 0).sum())
print("List price <= 0:          ", (sales["List_Price_EGP"] <= 0).sum())
print("COGS <= 0:                ", (sales["Estimated_COGS_EGP"] <= 0).sum())
print("Discount outside 0–1:     ", (~sales["Discount_Pct"].between(0, 1)).sum())
print("Negative refund:          ", (sales["Refund_Value_EGP"] < 0).sum())

# Gross profit CAN be negative (selling below cost) — not an error, but important
loss_lines = sales["Gross_Profit_EGP"] < 0
print(f"\nLines sold at a LOSS (profit < 0): {loss_lines.sum():,} ({loss_lines.mean():.2%})")

print("\nSummary:")
print(sales[["Quantity", "Discount_Pct", "Gross_Margin_Pct"] + money_cols].describe().T.round(2))

Quantity <= 0:             0
Quantity not whole number: 0
List price <= 0:           0
COGS <= 0:                 0
Discount outside 0–1:      0
Negative refund:           0

Lines sold at a LOSS (profit < 0): 4,876 (2.71%)

Summary:
                       count     mean      std      min     25%     50%      75%       max
Quantity            180000.0     9.23    14.74     1.00    1.00    1.00    12.00     60.00
Discount_Pct        180000.0     0.14     0.13     0.00    0.00    0.10     0.28      0.38
Gross_Margin_Pct    180000.0     0.29     0.13    -0.14    0.20    0.32     0.39      0.51
List_Price_EGP      180000.0   360.76    78.32   289.00  299.00  348.00   388.00    549.00
Gross_Sales_EGP     180000.0  3333.84  5501.20   289.00  348.00  549.00  3960.00  32940.00
Net_Sales_EGP       180000.0  2340.77  3669.40   216.75  310.40  494.10  2692.80  23716.80
Estimated_COGS_EGP  180000.0  1980.28  3270.57   155.11  200.02  316.82  2380.15  21628.41
Gross_Profit_EGP    180000.0   360.49 

In [15]:
# ============================================================
# 2.5 Money formulas — every amount must be consistent with the others
#   Gross_Sales  = List_Price x Quantity
#   Net_Sales    = Gross_Sales x (1 - Discount)
#   Gross_Profit = Net_Sales - COGS
#   Margin       = Gross_Profit / Net_Sales
# Small differences come from rounding to 2 decimals -> allow a tolerance
# ============================================================
def check_formula(name, calculated, recorded, tolerance):
    diff = (calculated - recorded).abs()
    bad = diff > tolerance
    print(f"{name:<14} off by more than {tolerance}: {bad.sum():>7,}   (max diff: {diff.max():.4f})")
    return bad

bad_gross  = check_formula("Gross_Sales",
                           sales["List_Price_EGP"] * sales["Quantity"],
                           sales["Gross_Sales_EGP"], 0.01)
bad_net    = check_formula("Net_Sales",
                           sales["Gross_Sales_EGP"] * (1 - sales["Discount_Pct"]),
                           sales["Net_Sales_EGP"], 0.05)
bad_profit = check_formula("Gross_Profit",
                           sales["Net_Sales_EGP"] - sales["Estimated_COGS_EGP"],
                           sales["Gross_Profit_EGP"], 0.05)
bad_margin = check_formula("Margin",
                           sales["Gross_Profit_EGP"] / sales["Net_Sales_EGP"],
                           sales["Gross_Margin_Pct"], 0.001)

any_bad = bad_gross | bad_net | bad_profit | bad_margin
if any_bad.any():
    display(sales.loc[any_bad, ["Invoice_ID", "Quantity", "List_Price_EGP", "Discount_Pct",
                                "Gross_Sales_EGP", "Net_Sales_EGP", "Estimated_COGS_EGP",
                                "Gross_Profit_EGP", "Gross_Margin_Pct"]].head(10))

Gross_Sales    off by more than 0.01:       0   (max diff: 0.0000)
Net_Sales      off by more than 0.05:       0   (max diff: 0.0000)
Gross_Profit   off by more than 0.05:       0   (max diff: 0.0000)
Margin         off by more than 0.001:       0   (max diff: 0.0000)


In [16]:
# ============================================================
# 2.6 Returns: flags, refund amounts, quality link
# ============================================================
yes_no = {"Yes": True, "No": False}
sales["Is_Returned"] = sales["Returned"].map(yes_no).astype("boolean")
sales["Is_Quality_Return"] = sales["Quality_Related_Return"].map(yes_no).astype("boolean")
print("Unmapped Returned:", sales["Is_Returned"].isna().sum(),
      "| Unmapped Quality:", sales["Is_Quality_Return"].isna().sum())
sales = sales.drop(columns=["Returned", "Quality_Related_Return"], errors="ignore")

ret = sales["Is_Returned"]

# 1) "No return" reason <-> not returned
print("\n========== Return_Reason x Is_Returned ==========")
print(pd.crosstab(sales["Return_Reason"], ret))

# 2) Refunds: 0 when not returned; never more than what the customer paid
print(f"\nNot returned but refund > 0:   {(~ret & (sales['Refund_Value_EGP'] > 0)).sum()}")
print(f"Returned but refund = 0:       {(ret & (sales['Refund_Value_EGP'] == 0)).sum()}")
print(f"Refund > Net_Sales:            {(sales['Refund_Value_EGP'] > sales['Net_Sales_EGP'] + 0.05).sum()}")
refund_ratio = (sales.loc[ret, "Refund_Value_EGP"] / sales.loc[ret, "Net_Sales_EGP"]).round(3)
print("Refund / Net_Sales for returns:", refund_ratio.value_counts().head(3).to_dict())

# 3) Quality-related returns = which reasons?
print("\n========== Return_Reason x Is_Quality_Return (returned lines) ==========")
print(pd.crosstab(sales.loc[ret, "Return_Reason"], sales.loc[ret, "Is_Quality_Return"]))

Unmapped Returned: 0 | Unmapped Quality: 0

========== Return_Reason x Is_Returned ==========
Is_Returned          False  True 
Return_Reason                    
Changed mind             0   1343
Color / appearance       0   1016
Delivery issue           0    677
Fabric / stitching       0   1405
No return           171457      0
Print issue              0   1052
Size / fit               0   3050

Not returned but refund > 0:   0
Returned but refund = 0:       0
Refund > Net_Sales:            0
Refund / Net_Sales for returns: {1.0: 6397, 0.75: 2146}

========== Return_Reason x Is_Quality_Return (returned lines) ==========
Is_Quality_Return   False  True 
Return_Reason                   
Changed mind         1343      0
Color / appearance   1016      0
Delivery issue        677      0
Fabric / stitching      0   1405
Print issue             0   1052
Size / fit           3050      0


In [17]:
# ============================================================
# 2.7 Types, column order, save
# ============================================================
before_mb = sales.memory_usage(deep=True).sum() / 1024**2

for col in ["Location_ID", "Channel", "Product_ID", "Size", "Payment_Method", "Return_Reason"]:
    sales[col] = sales[col].astype("category")
# Customer_ID: ~25,600 distinct values repeated across 180,000 rows -> category still helps
sales["Customer_ID"] = sales["Customer_ID"].astype("category")

column_order = ["Invoice_ID", "Order_Date", "Location_ID", "Channel", "Customer_ID",
                "Product_ID", "Size", "Quantity",
                "List_Price_EGP", "Discount_Pct", "Gross_Sales_EGP", "Net_Sales_EGP",
                "Estimated_COGS_EGP", "Gross_Profit_EGP", "Gross_Margin_Pct", "Payment_Method",
                "Is_Returned", "Return_Date", "Return_Reason", "Refund_Value_EGP", "Is_Quality_Return"]
extra_cols = [c for c in sales.columns if c not in column_order]
sales = sales[column_order + extra_cols]
if extra_cols:
    print("Extra columns kept at the end:", extra_cols)

after_mb = sales.memory_usage(deep=True).sum() / 1024**2
print(f"Memory: {before_mb:.1f} MB -> {after_mb:.1f} MB  ({(1 - after_mb / before_mb):.0%} smaller)")
print("Shape: before", sales_raw.shape, "-> after", sales.shape)

sales.to_parquet(CLEAN_DIR / "sales.parquet", index=False)
# The full CSV would be large -> save a 5,000-row random sample for Excel review
sales.sample(5_000, random_state=42).to_csv(CLEAN_DIR / "sales_sample.csv",
                                            index=False, encoding="utf-8-sig")
print("Saved: sales.parquet (full) + sales_sample.csv (5,000 rows for review)")

Memory: 35.5 MB -> 20.6 MB  (42% smaller)
Shape: before (180000, 30) -> after (180000, 21)
Saved: sales.parquet (full) + sales_sample.csv (5,000 rows for review)


In [18]:
# ============================================================
# 2.8 Headline numbers (first look at the business — full analysis comes next)
# ============================================================
# Bring customer type back from the customers file (one-line join)
ctype = sales["Customer_ID"].astype(str).map(cust.set_index("Customer_ID")["Customer_Type"].astype(str))

summary = sales.assign(Customer_Type=ctype).groupby("Channel", observed=True).agg(
    Lines        = ("Invoice_ID", "count"),
    Units        = ("Quantity", "sum"),
    Revenue      = ("Net_Sales_EGP", "sum"),
    Profit       = ("Gross_Profit_EGP", "sum"),
    Avg_Discount = ("Discount_Pct", "mean"),
    Avg_Qty      = ("Quantity", "mean"),
)
summary["Margin"]        = summary["Profit"] / summary["Revenue"]
summary["Revenue_Share"] = summary["Revenue"] / summary["Revenue"].sum()
summary["Profit_Share"]  = summary["Profit"] / summary["Profit"].sum()

pd.options.display.float_format = "{:,.2f}".format
display(summary)
pd.reset_option("display.float_format")

total_rev = sales["Net_Sales_EGP"].sum()
print(f"\nTotal net revenue: {total_rev:,.0f} EGP")
print(f"Total gross profit: {sales['Gross_Profit_EGP'].sum():,.0f} EGP  "
      f"(margin {sales['Gross_Profit_EGP'].sum() / total_rev:.1%})")
print(f"Return rate: {sales['Is_Returned'].mean():.1%} of lines | "
      f"Size/fit = {(sales['Return_Reason'] == 'Size / fit').sum() / sales['Is_Returned'].sum():.0%} of returns")

,Lines,Units,Revenue,Profit,Avg_Discount,Avg_Qty,Margin,Revenue_Share,Profit_Share
Channel,,,,,,,,,
Online,71559,99339,"32,629,574.45","11,384,070.74",0.09,1.39,0.35,0.08,0.18
Retail,61769,86174,"29,347,687.40","10,869,176.78",0.06,1.40,0.37,0.07,0.17
Wholesale,46672,1475076,"359,361,115.68","42,635,042.38",0.33,31.61,0.12,0.85,0.66



Total net revenue: 421,338,378 EGP
Total gross profit: 64,888,290 EGP  (margin 15.4%)
Return rate: 4.7% of lines | Size/fit = 36% of returns


### ✅ Sales_Transactions — Cleaning Summary
| Check / Decision | Result |
|---|---|
| Rows | 180,000; 1 line per invoice (`Invoice_ID` unique) |
| Duplicates (ignoring `Row_ID`) | 0 |
| Dates | 2023-10-01 → 2026-09-30, 100% converted |
| `Month_Key` = month of `Order_Date` | 100% match → dropped |
| `Returned` ↔ `Return_Date` | 100% consistent; returns within 0–21 days (median 11) |
| Referential integrity (Product / Location / Customer) | 0 missing IDs |
| Sales before customer `Join_Date` | 0 |
| Duplicated attributes vs. reference files | All identical (Product_Name differed only by spacing → 0 after tidying) → 7 columns dropped (normalization) |
| Channel ↔ Location | 1:1 (Online ↔ Online Store, Retail ↔ Sidi Bishr, Wholesale ↔ Wholesale Desk) |
| Channel ↔ Customer_Type | 1,961 wholesale-channel lines by Retail customers ("hidden traders"); 236 retail/online lines by Wholesale customers |
| Sizes | 0 sizes outside the product's catalog sizes; Kids = numeric, Women = letters |
| Numeric sanity | 0 invalid quantities, prices, COGS, discounts, refunds |
| Money formulas (Gross, Net, Profit, Margin) | 0 errors (max diff 0.0000) |
| Lines sold at a loss | 4,876 (2.71%), min margin −14% |
| Returns | Reasons, refunds and flags fully consistent; refund = 100% (6,397) or 75% (2,146); quality returns = Fabric/stitching + Print issue exactly |
| Dropped `Row_ID`, `Data_Class`; Yes/No → booleans; text → categories | Memory 80.8 MB → 20.6 MB (−75%) |

### 📊 Headline findings
| Channel | Revenue share | Profit share | Margin | Avg discount | Avg qty/line |
|---|---|---|---|---|---|
| Online | 8% | 18% | 35% | 9% | 1.4 |
| Retail | 7% | 17% | 37% | 6% | 1.4 |
| **Wholesale** | **85%** | **66%** | **12%** | **33%** | **31.6** |

- Total: 421.3M EGP revenue, 64.9M EGP gross profit (15.4% margin)
- Wholesale traders = 5.5% of customers but 85% of revenue → the wholesale desk IS the business
- Wholesale margin is 3× lower because of a 33% average discount → **≈ 5.4M EGP profit per 1 point of wholesale discount** (approx.)
- Return rate 4.7%; Size/fit = 36% of returns
- Sizes almost evenly split overall → size-mix insight must be validated per product/season
- 52 wholesale traders never ordered; wholesale pays mainly by bank transfer (75%)